# 完整的商業解決方案

## 現在將 Day 1 的專案提升到下一個層次

### 商業挑戰：

建立一個為公司製作宣傳手冊（Brochure）的產品，供潛在客戶、投資人與求職者使用。

我們會取得公司名稱與主要網站。

本 notebook 末尾有真實商業應用範例。

記住：若有問題或想法，我隨時在！請聯繫我。

In [ ]:
# 匯入
# 若失敗，請確認你在已啟用的環境中執行，命令提示字元應顯示 (llms)

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# 初始化與常數

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [ ]:
links = fetch_website_links("https://edwarddonner.com")
links

## 第一步：讓 GPT-5-nano 判斷哪些連結相關

### 呼叫 gpt-5-nano 讀取網頁上的連結，並以結構化 JSON 回覆。  
它應判斷哪些連結相關，並將相對連結如 "/about" 替換為 "https://company.com/about"。  
我們使用「一次示範提示詞（one shot prompting）」，在提示詞中提供回應範例。

這是 LLM 的絕佳用例，因為需要細膩理解。想像不用 LLM、靠解析與分析網頁來寫這段程式——會非常困難！

附註：有更進階的「Structured Outputs」技術，要求模型依規格回覆。我們會在第 8 週的自主代理式 AI 專案中涵蓋。

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## 第二步：製作手冊！

將所有細節組合成另一個給 GPT-5-nano 的提示詞

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# 或取消下方註解以使用更幽默的手冊版本——示範如何輕鬆融入「語氣」：

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # 超過 5,000 字元則截斷
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

## 最後——小改進

稍作調整，即可讓結果從 OpenAI 串流回傳，
呈現熟悉的打字機動畫

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# 為 Hugging Face 製作手冊時，試著將系統提示詞改為幽默版本：

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商業應用</h2>
            <span style="color:#181;">在本練習中，我們擴展 Day 1 程式碼，進行多次 LLM 呼叫並產生文件。

這或許是代理式 AI 設計模式的首個範例，因為我們結合多次 LLM 呼叫。第 2 週會有更多內容，第 8 週我們會大規模回到代理式 AI，建構完全自主的代理方案。

以這種方式產生內容是最常見的用例之一。與摘要類似，可應用於任何商業領域：撰寫行銷內容、依規格產生產品教學、建立個人化郵件內容等。探索內容產生如何應用於你的業務，並試著做概念驗證原型。看看 community-contributions 資料夾中其他學員的作品——非常多有價值的專案——太棒了！</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">進入第 2 週之前（非常有趣）</h2>
            <span style="color:#900;">請查看 week1 EXERCISE notebook，這是第 1 週結束的挑戰。這會讓你充分練習前沿 API，並為第 2 週做好準備。</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">三個實用資源提醒</h2>
            <span style="color:#f71;">1. 課程資源可在<a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">此處</a>取得。<br/>
            2. 我在 LinkedIn <a href="https://www.linkedin.com/in/eddonner/">這裡</a>，很樂意與修課學員連結！<br/>
            3. 我在試用 X/Twitter，帳號 <a href="https://x.com/edwarddonner">@edwarddonner<a>，希望有人教我怎麼用……  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">最後！有個特別請求</h2>
            <span style="color:#090;">
                我的編輯告訴我，學員在 Udemy 上為本課程評分會有巨大影響——這是 Udemy 決定是否推薦給他人的主要方式之一。若你能花一分鐘評分，我會非常感激！無論如何——若需要協助，請隨時聯繫 ed@edwarddonner.com。
            </span>
        </td>
    </tr>
</table>